<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Bölüm 2: Önceden Eğitilmiş Bir LLM ile Metin Üretmek

Bu not defterinde kullanılan paketler:

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.13
torch version: 2.10.0
tokenizers version: 0.21.4


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F01_raschka.webp?1" width="500px">

&nbsp;
## 2.1 Metin üretimi için LLM'lere giriş

- Bu bölümde kod yok
- LLM'ler metni nasıl üretir?
- Bu bölüm bir hazırlık bölümüdür: kitap boyunca kullanacağımız kodlama ortamını ve LLM'i kuruyoruz
- Ayrıca ilerideki bölümlerde kullanıp genişleteceğimiz metin üretme fonksiyonlarını da kodluyoruz

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F02_raschka.webp?1" width="300px">

- LLM (ve sinir ağı) akış şemaları geleneksel olarak yukarıdan aşağıya okunur ve çizilir

&nbsp;
## 2.2 Kodlama ortamını kurmak

- Bu kitabı okuyorsanız, daha önce Python ile kod yazmışsınızdır
- Halihazırda kurulu bir Python ortamınız varsa (Python 3.10 veya daha yenisi), bağımlılıkları kurmanın en basit yolu `pip` kullanmaktır:

In [2]:
#!pip install -r https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/refs/heads/main/requirements.txt

- Bu bölüm için bağımlılıklar elle de kurulabilir:

In [3]:
#!pip install torch>=2.10.0 tokenizers>=0.22.2 reasoning-from-scratch

- Benim tercih ettiğim yol, yaygın olarak önerilen [uv](https://docs.astral.sh/uv/) Python paket ve proje yöneticisini kullanmaktır
- `uv` kurmak için resmî web sitesinden işletim sisteminize uygun kurulumu çalıştırın: https://docs.astral.sh/uv/getting-started/installation/
- Ardından GitHub deposunu klonlayın:

In [4]:
#!git clone --depth 1 https://github.com/rasbt/reasoning-from-scratch.git

- `git` kurulu değilse, kaynak kod deposunu Manning web sitesinden ya da şu bağlantıya tıklayarak elle de indirebilirsiniz: https://github.com/rasbt/reasoning-from-scratch/archive/refs/heads/main.zip (indirdikten sonra arşivi açın)

- Terminalde `reasoning-from-scratch` klasörüne gidin
- Depo, `uv` aracının varsayılan olarak PyTorch ile uyumlu bir Python sürümü kullanması için bir `.python-version` dosyası içerir
- JupyterLab'ı başlatmak ve boş bir not defteri ya da bu bölümün not defterini açmak için `uv run jupyter lab` komutunu çalıştırın
- Bu komut ayrıca yerel bir sanal ortam kurar (genellikle `.venv/` içinde) ve `reasoning-from-scratch` klasöründeki `pyproject.toml` dosyasındaki tüm bağımlılıkları otomatik olarak yükler

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F03_raschka.webp?1" width="500px">

- Gerekirse ek kurulum ayrıntıları ve seçenekleri için [../02_setup-tips/python-instructions.md](../02_setup-tips/python-instructions.md) dosyasına bakın

&nbsp;
## 2.3 Donanım ihtiyaçlarını ve önerileri anlamak

- PyTorch'a yeniyseniz, [PyTorch in One Hour: From Tensors to Training Neural Networks on Multiple GPUs](https://sebastianraschka.com/teaching/pytorch-1h/) başlıklı eğitimimi okumanızı öneririm
- Önceki bölümü izlediyseniz PyTorch kurulu olmalı
- PyTorch kurulumunuzun GPU desteği olup olmadığını elle kontrol edin; makinenizde nelerin desteklendiğine bakın: 

In [5]:
import torch


print(f"PyTorch version {torch.__version__}")

if torch.cuda.is_available():
    print(f"CUDA/ROCm GPU: {torch.cuda.get_device_name(0)}")

elif torch.xpu.is_available():
    print(f"Intel GPU: {torch.xpu.get_device_name(0)}")

elif torch.backends.mps.is_available():
    print("Apple Silicon GPU")

else:
    print("Only CPU")

PyTorch version 2.10.0
Apple Silicon GPU


- Bölüme göre kod, varsa otomatik olarak bir NVIDIA (CUDA) GPU kullanır; yoksa CPU üzerinde çalışır (ya da belirli bir kısım veya bölüm için öneriliyorsa Apple Silicon GPU üzerinde)
- 2-4. bölümler bir CPU üzerinde makul bir sürede çalıştırılabilir
- 5-7. bölümlerdeki kod bir CPU üzerinde çok yavaş çalışır; bu bölümler için CUDA destekli bir GPU önerilir (kesin kaynak ihtiyaçlarına dair ayrıntılar o bölümlerde)
- Kişisel tercihim, kayıt ve doğrulama sürecinden sonra kullanıcılara ücretsiz hesaplama kredisi sunan [Lightning AI Studio](https://lightning.ai/); alternatif olarak [Google Colab](https://colab.research.google.com/) da iyi bir seçenektir

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F04_raschka.webp" width="500px">

- Gerekirse bulut hesaplama önerileri için [../02_setup-tips/gpu-instructions.md](../02_setup-tips/gpu-instructions.md) dosyasına bakın
- Ancak şimdilik GPU kullanmaya gerek yok; ilk bölümler GPU'suz donanımda sorunsuz çalışır

&nbsp;
## 2.4 LLM'ler için girdi metinlerini hazırlamak

- Bu bölümde bir tokenizer'ın nasıl kullanılacağını öğreniyoruz; onu, girdi metnini LLM'e girdi olacak şekilde token kimliği gösterimine dönüştürmek (kodlamak) için kullanıyoruz
- Ayrıca tokenizer'ı, LLM çıktısını insan tarafından okunabilir bir metin gösterimine geri dönüştürmek (kodunu çözmek) için de kullanıyoruz

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F05_raschka.webp?1" width="500px">

- Daha önce belirtildiği gibi, LLM'i ve tokenizer'ı sıfırdan uygulamak bu kitabın kapsamı dışındadır; kitap, mevcut bir LLM ve tokenizer üzerine akıl yürütme yöntemlerini sıfırdan uygulamaya odaklanır
- Bu kitapta, bir sonraki bölümde yükleyeceğimiz önceden eğitilmiş bir LLM ile çalışacağız; burada ona eşlik eden tokenizer'ı yüklüyoruz
- Temel LLM'i ve karşılık gelen tokenizer'ı sağlayan bir `reasoning_from_scratch` Python paketi hazırladım; bunu [`tokenizers`](https://github.com/huggingface/tokenizers) Python kütüphanesinin yardımıyla kodladım
- `reasoning_from_scratch` paket kodu bu kitabın ek kodunun bir parçasıdır ve 2.2 kısmındaki talimatlara göre zaten kurulmuş olmalıdır

- Ardından tokenizer dosyalarını indiriyoruz (bu, Qwen3 temel LLM'i için bir tokenizer'dır; ancak bununla ilgili ayrıntılar bir sonraki bölümde):

In [6]:
from reasoning_from_scratch.qwen3 import download_qwen3_small

download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")

- Şimdi tokenizer ayarlarını tokenizer dosyasından `Qwen3Tokenizer` içine yükleyebiliriz:

In [7]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer

tokenizer_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

- LLM'in kendisini henüz yüklemediğimiz için daha basit bir gidiş-dönüş yapacağız: metni token kimliklerine kodlayıp ardından kodunu çözerek dize gösterimine geri döndürüyoruz:

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F06_raschka.webp" width="500px">

In [8]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)

In [9]:
for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")

840 --> Ex
20772 --> plain
3460 -->  large
4128 -->  language
4119 -->  models
13 --> .


In [10]:
text = tokenizer.decode(input_token_ids_list)
print(text)

Explain large language models.


- `Qwen3Tokenizer` söz konusu olduğunda yaklaşık 151 bin benzersiz token (sözcük dağarcığı boyutu) vardır

- Token'lara ayırma konusunda ek kaynaklar:
  - [Build a Large Language Model (from Scratch)](https://mng.bz/M96o) 2. bölüm
  - [Implementing A Byte Pair Encoding (BPE) Tokenizer From Scratch](https://sebastianraschka.com/blog/2025/bpe-from-scratch.html)

&nbsp;
## 2.5 Önceden eğitilmiş modelleri yüklemek

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F07_raschka.webp" width="500px">

- Önceki bölümde tokenizer'ı yüklerken ima edildiği gibi, bu kitap Qwen3 0.6B kullanıyor; hangi açık ağırlıklı temel modeli kullanacağımı uzun uzun düşündükten sonra Qwen3'te karar kıldım, çünkü
  - Qwen3, bu yazının yazıldığı tarihte modelleme başarımı açısından önde gelen açık ağırlıklı modeldir
  - Qwen3 0.6B, Llama 3 1B'ye göre bellek açısından daha verimlidir
  - Hem bir temel model (akıl yürütme modeli geliştirmek için odaklandığımız model) hem de referans model olarak kullanabileceğimiz resmî bir akıl yürütme çeşidi mevcuttur
- (Kanonik yazımın "Qwen3" içinde boşluk içermediğini, buna karşılık "Llama 3" içinde boşluk bulunduğunu unutmayın)
- "Sıfırdan" ruhuna uygun olarak, Qwen3'ün saf PyTorch ile, herhangi bir dış LLM kütüphanesi bağımlılığı olmadan yazdığım bir yeniden uygulamasını kullanıyoruz; bu sıfırdan uygulama, özgün Qwen3 model ağırlıklarıyla uyumludur
- Ancak Qwen3 kod uygulamasını bu kitapta ele almayacağız; çünkü bu başlı başına bir kitap olurdu ([Build A Large Language Model (From Scratch)](https://github.com/rasbt/LLMs-from-scratch) kitabıma benzer şekilde); bunun yerine bu kitap (Build A Reasoning Model From Scratch) akıl yürütme yöntemlerini bir temel model (burada Qwen3) üzerine sıfırdan uygulamaya odaklanır
- Qwen3 model kodu için Ek C'ye bakın
- Akıl yürütme çeşidini ve daha büyük Qwen3 modellerini yüklemek için Ek D'ye bakın
- (Daha da) fazla ayrıntı için Qwen3 [GitHub deposuna](https://github.com/QwenLM/Qwen3) ve [teknik rapora](https://arxiv.org/abs/2505.09388) bakın

- Model, tüketici donanımında çalışabilmesi için bilinçli olarak küçük tutulmuştur (ama yine de çok yeteneklidir)
- CPU'da, NVIDIA GPU'larda (`"cuda"`), Apple Silicon GPU'larda (`"mps"`) ve Intel GPU'larda (`"xpu"`) sorunsuz çalışır; başarım ödünleşimleriyle ilgili ayrıntılar bu bölümün ilerleyen kısımlarında

In [11]:
def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using NVIDIA CUDA GPU")
        
        if enable_tensor_cores:
            major, minor = map(int, torch.__version__.split(".")[:2])
            # PyTorch 2.9 ve 2.10 sürümleri torch.compile içinde hâlâ eski TF32 ayarını okuyor.
            # Bkz. https://github.com/pytorch/pytorch/issues/166387
            # ve https://github.com/rasbt/reasoning-from-scratch/issues/256
            if (major, minor) >= (2, 11):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True

    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")

    elif torch.xpu.is_available():
        device = torch.device("xpu")
        print("Using Intel GPU")

    else:
        device = torch.device("cpu")
        print("Using CPU")

    return device

device = get_device()

Using Apple Silicon GPU (MPS)


- İlk okuyuşta kodu `"cpu"` üzerinde çalıştırmanızı öneriyorum; bu yüzden cihazı aşağıda sabit olarak belirliyoruz: 

In [12]:
# Recommended: Use CPU on the first run-through
device = torch.device("cpu")

- Ardından, yaklaşık 1,5 GB boyutundaki önceden eğitilmiş model ağırlıklarını içeren dosyayı indiriyoruz:

In [13]:
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

✓ qwen3/qwen3-0.6B-base.pth already up-to-date


- Yüklediğimiz Qwen3 0.6B modelinin mimari yapısı, LLM mimarilerine aşina okurlar için aşağıda gösterilmiştir; ancak bu kitap açısından bu mimariyi anlamanın **gerekli** ya da önemli olmadığını unutmayın; çünkü mimariyi değiştirmiyor, ilerideki bölümlerde onun üzerine akıl yürütme teknikleri ekliyoruz

- Qwen3 model mimarisini, bu kod deposunda yer alan [reasoning-from-scratch](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/qwen3.py) Python paketi için sıfırdan kodladım; kaynak kod Ek C'de de gösterilmiştir; ancak yine, bu yalnızca meraklılar için bir bonustur ve kitabın geri kalanını izlemek için bu iç yapılara bakmak ya da onları anlamak gerekmez

In [14]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

model_path = Path("qwen3") / "qwen3-0.6B-base.pth"

model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))

model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F08_raschka.webp" width="300px">

&nbsp;
## 2.6 Ardışık LLM metin üretme sürecini anlamak

- Bu bölümde, LLM'i metin üretmek için kullanabilmemiz adına basit bir sarmalayıcı fonksiyon kodluyoruz (bu fonksiyonu 4. bölümde ek işlevlerle genişleteceğiz)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F09_raschka.webp?1" width="500px">

- LLM'ler her seferinde bir kelime üretir:

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F10_raschka.webp?2" width="500px">

- Yukarıdaki şekil bir basitleştirmedir ve yalnızca yeni üretilen kelimeyi gösterir; aşağıdaki şekil ilk yinelemeye yakınlaşır:

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F11_raschka.webp" width="3b00px">

In [15]:
example = torch.tensor([1, 2, 3]) 
print(example)
print(example.unsqueeze(0))

tensor([1, 2, 3])
tensor([[1, 2, 3]])


In [16]:
example = torch.tensor([[1, 2, 3]]) 
print(example)
print(example.squeeze(0))

tensor([[1, 2, 3]])
tensor([1, 2, 3])


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F11_raschka.webp?2" width="300px">

In [17]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)
print(f"Number of input tokens: {len(input_token_ids_list)}")

input_tensor = torch.tensor(input_token_ids_list)
input_tensor_fmt = input_tensor.unsqueeze(0).to(device)

with torch.inference_mode():
    output_tensor = model(input_tensor_fmt)

output_tensor_fmt = output_tensor.squeeze(0)
print(f"Formatted Output tensor shape: {output_tensor_fmt.shape}")

Number of input tokens: 6
Formatted Output tensor shape: torch.Size([6, 151936])


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F12_raschka.webp" width="500px">

In [18]:
last_token = output_tensor_fmt[-1]
print(last_token)

tensor([ 7.3750,  2.0312,  8.0000,  ..., -2.5469, -2.5469, -2.5469],
       dtype=torch.bfloat16)


In [19]:
print(torch.argmax(last_token, dim=-1, keepdim=True))

tensor([20286])


In [20]:
print(tokenizer.decode([20286]))

 Large


In [21]:
example = torch.tensor([-2, 1, 3, 1])
print(torch.max(example))
print(torch.argmax(example))

tensor(3)
tensor(2)


&nbsp;
## 2.7 Asgari bir metin üretme fonksiyonu kodlamak


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F13_raschka.webp" width="500px">

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F14_raschka.webp?2" width="500px">

- `generate_text_basic_stream` fonksiyonu bu ardışık metin üretme sürecini uygular:

In [22]:
@torch.inference_mode()
def generate_text_basic_stream(
    model,
    token_ids,
    max_new_tokens, 
    eos_token_id=None
):
    model.eval()

    for _ in range(max_new_tokens):
        out = model(token_ids)[:, -1]
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        # Bir dizi-sonu token'ıyla karşılaşırsak dur
        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        yield next_token  # Yield each token as it's generated
        
        token_ids = torch.cat([token_ids, next_token], dim=1)

- Nasıl çalıştığını görmek için, basit bir `"Explain large language models in a single sentence."` istemine 100 token'lık bir yanıt üretmek üzere kullanalım (akıl yürütme kısımlarına ilerideki bölümlerde geliyoruz)
- Aşağıdaki kod yavaş olacak ve bilgisayarınıza bağlı olarak tamamlanması 1-3 dakika sürebilir (ilerideki kısımlarda hızlandıracağız) 

In [23]:
prompt = "Explain large language models in a single sentence."
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device
    ).unsqueeze(0)
max_new_tokens = 100


for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True  # Deactivates buffering so tokens are printed live
    )

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.<|endoftext|>Human language is a complex and dynamic system that has evolved over millions of years to enable effective communication and social interaction. It is composed of a vast array of symbols, including letters, numbers, and words, which are used to convey meaning and express thoughts and ideas. The evolution of language has

- LLM'in talimatı oldukça iyi izlediğine, ancak yanıtın `<|endoftext|>` sonrasında anlamsızlaştığına/konudan saptığına dikkat edin; bu token, eğitim sırasında farklı belgeler arasında ayraç olarak kullanılır
- LLM'i kullanırken, bu token'la karşılaştıktan sonra üretmeyi durdurmasını isteriz

In [24]:
print(tokenizer.encode("<|endoftext|>"))

[151643]


- Kolaylık olsun diye bu token kimliği bir tokenizer özniteliği olarak saklanır (eos = dizi sonu, end of sequence):

In [25]:
print(tokenizer.eos_token_id)

151643


- Bunu, LLM'e (ya da daha doğrusu `generate_text_basic_stream` fonksiyonuna) metin üretmeyi ne zaman durduracağını söylemek için kullanabiliriz

In [26]:
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id  # Use EOS token
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

- Yukarıdaki yanıt, kod CPU üzerinde çalıştırıldığında elde ettiğinizdir; üretilen metin cihaza bağlı olarak biraz farklılık gösterebilir

- Bu bölümü bitirip kodu nasıl hızlandırabileceğimize bakmadan önce, hesaplama başarımını izlemek için basit bir kıyaslama fonksiyonu uygulayalım

In [27]:
import warnings

def generate_stats(output_token_ids, tokenizer, start_time,
                   end_time):
    # tokenizer şu anda kullanılmıyor, geriye dönük uyumluluk için tutuluyor
    total_time = end_time - start_time
    print(f"\n\nTime: {total_time:.2f} sec")
    print(f"{int(output_token_ids.numel() / total_time)} tokens/sec")

    for name, backend in (("CUDA", getattr(torch, "cuda", None)),
                          ("XPU", getattr(torch, "xpu", None))):
        if backend is not None and backend.is_available():

            # Bu arka ucu gerçekten kullanıp kullanmadığımızı kontrol et
            device_type = output_token_ids.device.type
            if device_type != name.lower():
                warnings.warn(
                    f"{name} is available but tensors are on "
                    f"{device_type}. Memory stats may be 0."
                )
    
            # Destekleniyorsa eşitle (eşzamansız arka uçlar için önemli)
            if hasattr(backend, "synchronize"):
                backend.synchronize()
            
            max_mem_bytes = backend.max_memory_allocated()
            max_mem_gb = max_mem_bytes / (1024 ** 3)
            print(f"Max {name} memory allocated: {max_mem_gb:.2f} GB")
            backend.reset_peak_memory_stats()

In [28]:
import time

start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # Collect generated tokens

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Time: 1.39 sec
29 tokens/sec


&nbsp;
## 2.8 KV önbelleği ile daha hızlı çıkarım

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F15_raschka.webp?2" width="500px">

- Bu kitaptaki kodun okunabilirliği öne çıkardığını ve optimizasyonlar üzerine başlı başına ayrı bir kitap yazılabileceğini unutmayın
- Burada "KV önbellekleme" adı verilen bir mühendislik hilesine bakıyoruz (KV, LLM'in dikkat mekanizmasındaki anahtarları (key) ve değerleri (value) belirtir)
- Bu terimlere yabancıysanız endişelenmeyin; bilmeniz gereken tek şey, her yinelemede yeniden kullanılan ara değerleri saklamanın (önbelleğe almanın) bir yolu olduğudur

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F16_raschka.webp" width="500px">

- KV önbelleklemenin mekaniğine dair daha fazla ayrıntı için [Understanding and Coding the KV Cache in LLMs from Scratch](https://magazine.sebastianraschka.com/p/coding-the-kv-cache-in-llms) yazıma bakın
- Aşağıda, `generate_text_basic_stream` fonksiyonunun KV önbelleği kullanan değiştirilmiş bir sürümü yer alıyor

In [29]:
from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode()
def generate_text_basic_stream_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None
):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])  # New
    model.reset_kv_cache()                           # New

    out = model(token_ids, cache=cache)[:, -1]
    for _ in range(max_new_tokens):
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        yield next_token
        out = model(next_token, cache=cache)[:, -1]

- Kullanımı öncekine benzer:

In [30]:
start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # Collect generated tokens

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Time: 0.84 sec
49 tokens/sec


- Görüldüğü gibi KV önbellekleme, Mac Mini M4 CPU üzerinde üretimi belirgin biçimde hızlandırıyor

&nbsp;
## 2.9 PyTorch model derlemesi ile daha hızlı çıkarım

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F17_raschka.webp?2" width="500px">

- Model çıkarımını (metin üretimini) büyük ölçüde hızlandıran bir başka teknik de `torch.compile` kullanmaktır
- Kullanımı basittir; modele yalnızca `torch.compile` uygularız (ek seçenekler için [belgelere](https://docs.pytorch.org/docs/stable/torch.compiler_api.html) bakın)

In [31]:
major, minor = map(int, torch.__version__.split(".")[:2])
if (major, minor) >= (2, 8):
    # Bu, PyTorch 2.8 ve daha yeni sürümlerde model yeniden derlemelerinin
    # tekrar tetiklenmesini önler
    # if the model contains code like self.pos = self.pos + 1
    torch._dynamo.config.allow_unspec_int_on_nn_module = True

model_compiled = torch.compile(model)

# "mps" cihazlarında torch.compile ile sorun yaşıyor ve bir InductorError alıyorsanız,
# PyTorch 2.9 veya daha yenisini kullandığınızdan emin olun

---

**Windows notu 1**

- Derleme Windows'ta zahmetli olabilir
- `torch.compile()`, çekirdekleri JIT ile derleyen ve çalışan bir C/C++ araç zinciri gerektiren Inductor'ı kullanır
- CUDA için Inductor ayrıca, topluluk paketi `triton-windows` aracılığıyla erişilebilen Triton'a bağımlıdır
  - `cl not found` hatası görürseniz, [Visual Studio Build Tools'u "C++ workload" ile kurun](https://learn.microsoft.com/en-us/cpp/build/vscpp-step-0-installation?view=msvc-170) ve Python'u "x64 Native Tools" isteminden çalıştırın
  - CUDA ile `triton not found` hatası görürseniz `triton-windows` kurun (örneğin `uv pip install "triton-windows<3.4"`).
- CPU için bir okur, şu [Windows için PyTorch Inductor rehberini](https://docs.pytorch.org/tutorials/unstable/inductor_windows.html) izlemeyi önerdi
  - Burada, bir UTF-8 hatasından kaçınmak için Visual Studio 2022 kurulumunda İngilizce dil paketini kurmak önemlidir
  - Ayrıca kodun bir not defteri yerine "Visual Studio 2022 Developer Command Prompt" üzerinden çalıştırılması gerektiğini unutmayın
- Bu kurulum zahmetli gelirse derlemeyi atlayabilirsiniz; **derleme isteğe bağlıdır ve tüm kod örnekleri onsuz da sorunsuz çalışır**

**Windows notu 2**

- Okurlar, Windows'ta `torch.compile` varsayılan ayarlarla çalıştırıldığında hız kazancı olmadığını bildirdi; ancak `torch.compile` fonksiyonunu `"max-autotune"` kipiyle çalıştırmak 2 kat hız kazancı sağladı: `torch.compile(model, mode="max-autotune")`

---

- İlk yineleme, başlangıçtaki derleme ve optimizasyonu yaptığı için biraz yavaş olabilir; bu nedenle metin üretimini birden çok kez tekrarlıyoruz
- Önce önbelleksiz sürümle başlayalım (bu biraz yavaş olabilir ve xx dakika sürebilir)

In [32]:
for i in range(3):

    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # Collect generated tokens
    
    end_time = time.time()
    

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

    print(f"\n{30*'-'}\n")

W0213 17:02:09.090000 73246 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays.

Warm-up run


Time: 27.15 sec
1 tokens/sec

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays.

Timed run 1:


Time: 0.82 sec
42 tokens/sec

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays.

Timed run 2:


Time: 0.82 sec
42 tokens/sec

------------------------------



- Yukarıda görebileceğimiz gibi, saniyede 5 token ile bu, öncekinden (saniyede 4 token) yalnızca çok az daha hızlı
- Şimdi KV önbellekli sürümün ne kadar iyi olduğuna bakalım

In [33]:
for i in range(3):
    
    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream_cache(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # Collect generated tokens
    
    end_time = time.time()

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(
        output_token_ids_tensor, tokenizer, start_time, end_time
    )

    print(f"\n{30*'-'}\n")

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Warm-up run


Time: 45.89 sec
0 tokens/sec

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Timed run 1:


Time: 0.48 sec
84 tokens/sec

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Timed run 2:


Time: 0.45 sec
90 tokens/sec

------------------------------



- Görüldüğü gibi derleme, kayda değer bir 2 kat hız kazancı sağladı (saniyede 64 token'a karşı saniyede 30 token)
- Aşağıda ek sonuçları içeren bir tablo yer alıyor

| Model      | Kip               | Donanım              | Token/sn      | GPU Belleği (VRAM) |
|------------|-------------------|----------------------|---------------|-------------------|
| Qwen3Model | Normal            | Mac Mini M4 CPU      | 5             | -                 |
| Qwen3Model | Normal derlenmiş  | Mac Mini M4 CPU      | 5             | -                 |
| Qwen3Model | KV önbelleği      | Mac Mini M4 CPU      | 29            | -                 |
| Qwen3Model | KV önbelleği derlenmiş | Mac Mini M4 CPU | 68            | -                 |
|            |                   |                      |               |                   |
| Qwen3Model | Normal            | Mac Mini M4 GPU      | 27            | -                 |
| Qwen3Model | Normal derlenmiş  | Mac Mini M4 GPU      | 43            | -                 |
| Qwen3Model | KV önbelleği      | Mac Mini M4 GPU      | 41            | -                 |
| Qwen3Model | KV önbelleği derlenmiş | Mac Mini M4 GPU | 71            | -                 |
|            |                   |                      |               |                   |
| Qwen3Model | Normal            | NVIDIA H100 GPU      | 51            | 1.55 GB           |
| Qwen3Model | Normal derlenmiş  | NVIDIA H100 GPU      | 164           | 1.81 GB           |
| Qwen3Model | KV önbelleği      | NVIDIA H100 GPU      | 48            | 1.52 GB           |
| Qwen3Model | KV önbelleği derlenmiş | NVIDIA H100 GPU | 141           | 1.81 GB           |
|            |                   |                      |               |                   |
| Qwen3Model | Normal            | NVIDIA DGX Spark GPU | 74            | 1.53 GB           |
| Qwen3Model | Normal derlenmiş  | NVIDIA DGX Spark GPU | 103           | 1.49 GB           |
| Qwen3Model | KV önbelleği      | NVIDIA DGX Spark GPU | 68            | 1.47 GB           |
| Qwen3Model | KV önbelleği derlenmiş | NVIDIA DGX Spark GPU | 98        | 1.47 GB           |

- Yukarıdaki NVIDIA DGX Spark bir GB10 (Blackwell) GPU kullanır
- Tüm örnekleri tek bir istemle (yani yığın boyutu 1 ile) çalıştırdığımızı unutmayın; yığınlı çıkarımı merak ediyorsanız Ek E'ye bakın

&nbsp;
## Özet

- Bu bölümde kod yok